# 2. LangChain Prompt Template

## 학습 목표

1. 프롬프트와 프롬프트 템플릿이 필요한 이유를 설명할 수 있습니다.
2. `PromptTemplate`으로 단일 문자열 프롬프트를 만듭니다.
3. `ChatPromptTemplate`으로 System / Human / AI 역할이 구분된 대화형 프롬프트를 만듭니다.
4. Few-shot 예시로 답변 형식과 말투를 고정합니다.

**기준 버전**: LangChain 1.x, `langchain-openai` 1.x, 모델 `gpt-6-luna`

## 1. 프롬프트(Prompt)란?

- **정의**: LLM에게 "무엇을, 어떻게 해 달라"고 지시하는 자연어 명령문입니다.
- **본질**: 사람에게 질문할 때도 명확하게 요청해야 원하는 답을 얻듯, LLM에게도 프롬프트는 "명령어이자 대화의 시작점"입니다.

> 💡 **같은 주제, 다른 결과**
> - 일반 지시: "고양이에 대해 설명해 줘."
> - 구체적 지시: "고양이를 5살 아이가 바로 이해할 수 있게 쉬운 비유를 들어 설명해 줘."
>
> 지시어 하나로 답변의 깊이와 어조(Tone & Manner)가 완전히 달라집니다. 프롬프트 설계(프롬프트 엔지니어링)가 답변의 질을 결정합니다.

### PromptTemplate을 사용하는 이유

LangChain에서는 프롬프트를 코드에 직접 써 넣는 대신 **템플릿**으로 관리합니다.

- **일관성 유지**: 답변의 구조와 품질을 일정하게 유지합니다.
- **재사용성**: 하나의 틀에 다양한 입력값을 넣어 재사용합니다.
- **유지보수성**: 역할(System, Human, AI)과 지침을 분리해 관리합니다.

| 종류 | 용도 |
|---|---|
| `PromptTemplate` | 단일 문자열 프롬프트 |
| `ChatPromptTemplate` | 역할이 구분된 대화형 프롬프트 (Chat 모델 표준) |

### 실습 환경 준비

터미널에서 필요한 패키지를 설치하고, 프로젝트 최상위 폴더의 `.env` 파일에 API 키와 모델명을 저장합니다.

```text
OPENAI_API_KEY="YOUR_OPENAI_API_KEY"
OPENAI_DEFAULT_MODEL="gpt-6-luna"
```

> API 키는 코드 셀이나 GitHub에 직접 입력하지 마세요.

```bash
uv add langchain langchain-openai python-dotenv
```

In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")

# reasoning_effort="none": 수업 예제는 빠른 응답과 낮은 비용을 위해 추론 기능을 끕니다.
llm = ChatOpenAI(model=default_model, reasoning_effort="none")
print("사용 모델:", default_model)

> **gpt-6-luna 사용 시 참고**
> - `reasoning_effort`는 `none` / `low` / `medium`(기본) / `high` / `xhigh` / `max`를 지원합니다. 강의 예제는 `none`으로 통일합니다.
> - `temperature`는 추론이 꺼져 있을 때(`reasoning_effort="none"`)만 적용됩니다.
> - 모델명·요금·지원 파라미터는 바뀔 수 있으므로 수업 전에 [OpenAI 모델 문서](https://developers.openai.com/api/docs/models/gpt-6-luna)를 확인하세요.

## 2. 단일 문자열 프롬프트 - PromptTemplate

- 단일 문자열 프롬프트를 만드는 가장 간결한 방법입니다.
- 문자열 안의 `{변수}`를 자동으로 찾아내므로 변수 목록을 따로 지정할 필요가 없습니다.

### 2-1. PromptTemplate.from_template()

In [ ]:
from langchain_core.prompts import PromptTemplate

# 1. 템플릿 생성 ({country} 변수 자동 감지)
template = PromptTemplate.from_template("{country}의 수도는 어디이고, 대표적인 명소 2곳은 어디인가요?")

# 2. LCEL 체인 구성
chain = template | llm

# 3. 체인 실행 (딕셔너리로 변수 전달)
response = chain.invoke({"country": "프랑스"})
print(response.content)

#### 왜 딕셔너리를 사용하나요?

- 템플릿 안의 **변수 이름**과 **실제 값**을 짝지어 전달해야 합니다.
- "이 이름의 변수에는 이 값을 넣어 줘"라는 Key-Value 매칭에 가장 명확한 구조가 딕셔너리입니다.
- LangChain 프롬프트 템플릿은 입력으로 **항상 딕셔너리**를 받도록 약속되어 있습니다.

#### LCEL 체인의 작동 순서 (`chain = template | llm`)

1. 입력: `{"country": "프랑스"}`
2. 템플릿 치환: `{country}` 자리를 "프랑스"로 바꿔 완성된 프롬프트 생성
3. LLM 전달: 완성된 프롬프트로 답변 생성

> 템플릿이 어떻게 완성되는지는 `template.invoke({...})`로 미리 확인할 수 있습니다.

In [ ]:
print(template.invoke({"country": "프랑스"}).to_string())

### 2-2. 변수가 2개 이상인 경우

딕셔너리의 순서가 바뀌어도 이름으로 매칭되므로 `food`에는 비빔밥, `city`에는 서울이 정확히 들어갑니다.

In [ ]:
template = PromptTemplate.from_template("{city}에서 가장 맛있는 {food}에 대해 알려 줘")
chain = template | llm

response = chain.invoke({"food": "비빔밥", "city": "서울"})
print(response.content)

### 2-3. 실습하기

1. **스포츠 팀 정보 조회**: 프로축구 / 3개 팀, 프로야구 / 5개 팀
2. **마케팅 문구 생성하기** (`product` / `target` / `tone`): 무설탕 디저트 / 2030 직장인 / 위트 있고 감각적인
3. **요리 레시피 가져오기**: 카레

In [ ]:
# 실습 1) 스포츠 팀 정보 조회 - 여기에 작성하세요.

#### 정답 예시 1) 스포츠 팀 정보 조회

In [ ]:
template = PromptTemplate.from_template("{league}의 대표적인 인기 팀 {count}개를 알려 주세요.")
chain = template | llm

res1 = chain.invoke({"league": "KBO 프로야구", "count": 5})
res2 = chain.invoke({"league": "K리그 프로축구", "count": 3})

print(res1.content)
print(res2.content)

#### 정답 예시 2) 마케팅 문구 생성하기

In [ ]:
template = PromptTemplate.from_template(
    "{target}을(를) 대상으로 한 {product}의 홍보 슬로건 3가지를 작성해 주세요. 톤앤매너는 {tone} 느낌으로 작성해 주세요."
)
chain = template | llm

res = chain.invoke({
    "product": "무설탕 디저트",
    "target": "2030 직장인",
    "tone": "위트 있고 감각적인",
})
print(res.content)

#### 정답 예시 3) 요리 레시피 가져오기

In [ ]:
template = PromptTemplate.from_template("{dish} 기본 레시피를 재료와 만드는 방법으로 구분해서 알려 주세요.")
chain = template | llm

response = chain.invoke({"dish": "카레"})
print(response.content)

## 3. 대화형 프롬프트 - ChatPromptTemplate

Chat 모델을 다룰 때 표준으로 사용하는 방식입니다. 단순 문자열이 아니라 **System / Human / AI 역할을 분리**해 프롬프트를 구성합니다.

### 3-1. LLM이 맥락(Context)을 이해하는 방식

| 역할 (Role) | 주요 역할 및 용도 |
|---|---|
| System | 모델의 역할, 페르소나, 규칙, 말투, 출력 형식 정의 (상위 지침) |
| Human | 사용자의 실제 질문이나 요청 (변수 주입) |
| AI | 모델의 답변. Few-shot 예시로 원하는 답변 형태를 보여 줄 때도 사용 |

세 가지를 함께 쓰면 모델은 "지금 나는 친절한 선생님이고, 방금 '인공지능이 뭐냐'는 질문을 들었구나"라고 이해한 뒤 더 일관된 답변을 합니다.

### 3-n. ChatPromptTemplate.from_template()
- 시스템 메세지나 대화 기록이 필요 없는 간단한 질문일 경우

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

## from_template: human 메시지 하나로 자동 변환
prompt = ChatPromptTemplate.from_template(
    "서울 남대문시장에서 가장 맛있는 길거리 음식은 무엇인가요?"
)


In [ ]:
prompt = ChatPromptTemplate.from_template(
    "서울 {place}에서 가장 맛있는 길거리 음식은 무엇인가요?"
)

chain = prompt | llm
response = chain.invoke({"place": "남대문시장"})
print(response.content)


### 3-2. ChatPromptTemplate.from_messages()
- 역할을 나누거나 멀티턴 대화가 필요할 때 사용

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 친절한 한국어 챗봇이야. 답변은 간단히 3줄로 해 줘."),
    ("human", "{question}"),
])

chain = prompt | llm

response = chain.invoke({"question": "PromptTemplate과 ChatPromptTemplate의 차이가 뭐야?"})
print(response.content)

- `from_messages()`는 `("역할", "내용")` 튜플을 `SystemMessage`, `HumanMessage` 등의 객체로 자동 변환해 줍니다.
- 메시지 클래스를 일일이 import하지 않아도 되고, 위에서 아래로 대화 순서가 한눈에 보입니다.

#### 튜플형 리스트 구조

```python
[
    ("system", "시스템 지침 내용"),
    ("human", "사용자 질문 내용"),
    ("ai", "AI의 예시 답변 내용"),
]
```

- 대괄호 `[ ]`(리스트): 순서가 있는 대화 흐름을 담습니다.
- 소괄호 `( )`(튜플): `(역할, 내용)`을 한 묶음으로 고정합니다.
  - 첫 번째 요소(역할): `"system"`, `"human"`(`"user"`), `"ai"`(`"assistant"`)
  - 두 번째 요소(내용): 프롬프트 문자열 (변수 포함 가능)

> 멀티턴 대화에서는 `MessagesPlaceholder("history")`로 이전 대화 기록을 프롬프트 중간에 끼워 넣을 수 있습니다. → **3. Memory**에서 다룹니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 서울의 음식과 문화 전문가입니다."),
    MessagesPlaceholder("history"),   # 이전 대화가 들어갈 자리
    ("human", "{question}"),
])

chain = prompt | ChatOpenAI(model=default_model) | StrOutputParser()

history = []

# 1턴
q1 = "서울을 대표하는 음식 하나만 알려줘."
a1 = chain.invoke({"history": history, "question": q1})
history += [HumanMessage(q1), AIMessage(a1)]
print(a1)   

# # 2턴: 앞의 대화를 기억하고 답변
q2 = "그 음식을 먹을 수 있는 식당도 추천해줘."
a2 = chain.invoke({"history": history, "question": q2})
history += [HumanMessage(q2), AIMessage(a2)]
print(a2)

In [ ]:
# print(prompt.invoke({"history": history, "question": q1}).to_string())
# print('-'*30)
print(prompt.invoke({"history": history, "question": q2}).to_string())

### 3-3. 실습하기

1. **파이썬 코드에 주석 달기** (변수 1개)
   - system: Python 개발자 / 목적: 전달받은 코드에 친절한 주석 추가
   ```python
   number = [1, 2, 3, 4, 5]
   squared = [i**2 for i in number]
   print(squared)
   ```
2. **다국어 비즈니스 이메일 변환기** (변수 2개)
   - system: 비즈니스 통번역가 (언어는 변수로 전달)
   - 번역할 내용: "내일 회의 시간 2시로 변경 가능한가요? 확인 후 답장 부탁드립니다."
3. **맞춤형 페르소나 챗봇** (변수 3개)
   - 페르소나: 조선 시대 세종대왕 / 말투: 위엄 있고 엄격한 왕
   - 질문: "요즘 공부가 너무 하기 싫은데 어떻게 해야 할까요?"

In [ ]:
# 실습 1) 코드에 주석 달기 - 여기에 작성하세요.

#### 정답 예시 1) 파이썬 코드에 주석 달기

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 파이썬 튜터입니다. 입력된 코드의 각 줄에 초보자가 이해하기 쉬운 주석을 달아 주세요."),
    ("human", "다음 코드에 주석을 달아 주세요.\n{code}"),
])
chain = prompt | llm

sample_code = '''
number = [1, 2, 3, 4, 5]
squared = [i**2 for i in number]
print(squared)
'''

response = chain.invoke({"code": sample_code})
print(response.content)

#### 정답 예시 2) 다국어 비즈니스 이메일 변환기

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 전문 글로벌 비즈니스 커뮤니케이터입니다. 전달받은 내용을 {language}로 번역해 주세요."),
    ("human", "다음 내용을 정중한 이메일로 바꿔 주세요.\n{text}"),
])
chain = prompt | llm

response = chain.invoke({
    "language": "영어",
    "text": "내일 회의 시간 2시로 변경 가능한가요? 확인 후 답장 부탁드립니다.",
})
print(response.content)

#### 정답 예시 3) 맞춤형 페르소나 챗봇

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {persona}입니다. 모든 답변은 {tone} 말투로 작성하세요."),
    ("human", "{question}"),
])
chain = prompt | llm

response = chain.invoke({
    "persona": "조선 시대 세종대왕",
    "tone": "위엄 있고 엄격한 왕의",
    "question": "요즘 공부가 너무 하기 싫은데 어떻게 해야 할까요?",
})
print(response.content)

## 4. Few-shot 프롬프트

지침만 주는 것(Zero-shot)보다 **원하는 답변 예시를 1~3개 보여 주는 것**이 답변 형식을 훨씬 정확하게 유도합니다.

> "이렇게 답해 줘"라고 말하는 대신 "이런 예시들이 있어 → 이제 같은 형식으로 답해 줘"라고 보여 주는 방식입니다.

> 💡 **Few-shot이 특히 필요한 경우**
> - 출력 형식 통제: JSON 같은 정해진 포맷이 필요할 때
> - 답변 스타일 고정: 특정 말투, 일정한 문장 구조를 유지해야 할 때
> - 분류·판단 정확도 향상: 감정 분석, 단계별 문제 풀이 등

### 4-1. ChatPromptTemplate으로 Few-shot 구성

| 메시지 역할 | 역할 명칭 | 주요 용도 |
|---|---|---|
| System | 지침 (Instruction) | 페르소나와 전체 작업 규칙 정의 |
| Human / AI | 예시 (Examples) | "이런 질문에는 이렇게 답하라"는 모범 답안 1~2개 |
| Human | 입력 (Input) | `{input}` 변수가 들어갈 실제 질문 자리 |

먼저 AI 메시지로 대화 흐름을 미리 깔아 두는 예시입니다.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 베테랑 여행 가이드로, 고객에게 최적의 맞춤 여행 일정을 도와줍니다."),
    ("human", "{when}에 {place}로 {travel}을 계획하고 있습니다."),
    ("ai", "제가 바로 {place} 전문 여행 가이드입니다. 어떤 도움이 필요하신가요?"),
    ("human", "{question}"),
])
chain = prompt | llm

response = chain.invoke({
    "when": "여름",
    "place": "동남아",
    "travel": "가족여행",
    "question": "해당 지역의 여행지 5군데를 추천해 주세요.",
})
print(response.content)

Human/AI 예시 쌍으로 **답변 형식**을 학습시키는 예시입니다.

In [ ]:
few_shot_prompt = ChatPromptTemplate.from_messages([
    # 지침 (System)
    ("system", "문장을 긍정 또는 부정으로 분류하세요."),
    # 예시 1
    ("human", "문장: 이 영화는 최고였다."),
    ("ai", "답: 긍정"),
    # 예시 2
    ("human", "문장: 이 음식은 형편없다."),
    ("ai", "답: 부정"),
    # 실제 입력
    ("human", "문장: {input}"),
])
chain = few_shot_prompt | llm

response = chain.invoke({"input": "이 서비스는 정말 만족스럽다."})
print(response.content)

### 4-2. 실습하기

1. **사투리를 표준어로 번역하기**
   - system: 사투리로 작성된 문장을 표준어로 번역하고, 핵심 요청 사항을 정리
   - 예시 human: "아이고 모가 목말라 죽겠는데 물 한 바가지 퍼 온나."
   - 예시 ai: "표준어: 목이 너무 마르니 물 한 바가지 가져다주세요.\n핵심 요청: 물 가져오기"
   - 질문: "오늘 날씨도 억수로 좋은데 퍼뜩 밖으로 나갑시다."
2. **감정 분석 및 사유 정리 (JSON 출력 고정)**
   - system: 고객 리뷰를 분석하여 감정(positive/negative)과 핵심 이유를 JSON 형식으로 추출
   - 예시 2개를 보여 주면 LLM이 항상 같은 JSON 구조로 답합니다.
   - 질문: "디자인은 예쁜데 생각보다 사이즈가 많이 작아서 교환해야 할 것 같아요."

> ⚠️ **프롬프트 안의 `{ }` 사용법**
> 템플릿에서 `{ }`는 변수 기호입니다. JSON 예시처럼 중괄호 문자 자체를 넣으려면 `{{`와 `}}`로 두 번 써야 오류가 나지 않습니다.

In [ ]:
# 실습 1) 사투리 번역 - 여기에 작성하세요.

In [ ]:
# 실습 2) 감정 분석 및 사유 정리 (JSON)